# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "1"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 81920

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Think deeply and explore multiple approaches before committing to a solution. "
    "Solve the problem ONLY by showing necessary calculation steps. "
    "After solving, verify your answer by checking it against the original problem. "
    "If your verification fails, try a different approach. "
    "Take as much time as needed to find the correct answer. "

    # -- counting answers --
    "COUNTING YOUR ANSWERS: "
    "Count the number of [ANS] placeholders in the problem. "
    "Your boxed answer MUST contain EXACTLY that many comma-separated items inside a single \\boxed{}. "
    "If there are no [ANS] placeholders, give one complete answer inside \\boxed{}. "
    "Do NOT skip any sub-part. Do NOT box only the last value. "

    # -- single [ANS] with multiple components --
    "SINGLE [ANS] WITH MULTIPLE COMPONENTS: "
    "If there is only 1 [ANS] but the answer has multiple components (multiple zeros, "
    "multiple letters, a vector, a solution set), combine them into ONE item: "
    "- For numerical tuples/coordinates/solution sets: use parentheses e.g. (2, -2) or (80, -160, 320). "
    "- For letter combinations (multi-select): concatenate with NO spaces or commas e.g. BEF not B, E, F. "
    "- NEVER split a single [ANS] into multiple comma-separated items in the box. "
    "If there are 2 [ANS] placeholders, give exactly 2 comma-separated items — one per [ANS]. "

    # -- format rules --
    "FORMAT RULES — follow carefully: "
    "1. NEVER include trailing zeros: write 0.34 not 0.340, write 2.5 not 2.50, write 35.0000 as 35. "
    "2. NEVER include currency symbols: write 38800 not $38,800 or \\$38,800.00. "
    "3. NEVER include units inside \\boxed{} unless the problem explicitly asks for them. "
    "4. Confidence intervals MUST use parentheses: e.g. (81.03, 84.41). "
    "   If the question gives separate UCL=[ANS] LCL=[ANS], give them separately, not as a pair. "
    "5. Coordinate pairs and solution sets that form a SINGLE answer use parentheses: (-1, -3), (2, -2). "
    "6. When the question asks for multiple zeros/solutions in ONE [ANS], wrap them ALL in parentheses: (2, -2, 5). "
    "7. YES/NO/TRUE/FALSE answers should be capitalized: YES, NO, TRUE, FALSE. "
    "8. If question asks for answer as a percentage like '12%', include the % symbol. "
    "9. If question says 'express in scientific notation', use E notation: 1.67E-24 not 1.67*10^-24. "
    "10. For infinity in intervals, ALWAYS write infinity not \\infty or infty or inf: "
    "    write (-8, infinity) not (-8, \\infty) or (-8, infty) or (-inf, 5). "
    "11. Make sure all LaTeX expressions inside \\boxed{} have complete matching braces. "

    # -- precision rules --
    "PRECISION RULES: "
    "If a specific sub-question says 'round to X decimal places' or 'correct to X decimal places' "
    "or 'round to whole number', apply that rounding ONLY to that specific sub-answer. "
    "'At least X decimal places' means give MORE precision than X, not exactly X — give full precision. "
    "For ALL other answers: give the FULL precision value you computed. "
    "If you computed 143.224229233795, box 143.224229233795 not 143.22. "
    "NEVER round an answer unless that specific sub-question explicitly asks for it. "

    # -- exact vs decimal --
    "EXACT vs DECIMAL: "
    "If the problem says 'exact', 'do not approximate', or 'express as expression', "
    "give the exact symbolic expression, not a decimal. "
    "e.g. \\boxed{\\frac{\\ln(245/7)}{\\ln(3)}} not \\boxed{3.298}. "
    "If the problem gives numerical inputs and asks for a number without saying exact, "
    "give full-precision decimal. "

    # -- simplification --
    "Do not simplify your final answer unless asked. "
    "The grader accepts numerically equivalent expressions."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "First solve the problem without reading the answers. "
    "Then compare your answer with the options and pick the corresponding one. "
    "If none of the options align with yours then VERIFY your calculations or choose a different approach. "
    "Take as much time as needed to find the correct answer. "
    "Select the option that makes the MOST sense and is MOST PLAUSIBLE given the question. "

    # -- equivalent options --
    "If two options appear mathematically equivalent, always choose the SIMPLIFIED form. "
    "For example prefer cos^6(x) over cos^5(x)*cos(x), prefer 1/2 over 2/4. "

    # -- structured conclusion --
    "Before writing your final boxed answer, you MUST write the line: "
    "'The answer is option X' where X is the letter from the given options. "
    "Then immediately box that same letter: \\boxed{X}. "
    "The letter in 'The answer is option X' and in \\boxed{X} MUST be identical. "
    "NEVER box a variable name from your working — only box the option letter."
)

FEW_SHOT_MATH = """Here are examples of how to solve free-form problems correctly and incorrectly:

Example 1 (CORRECT — count [ANS], give full precision):
Problem: Find the sum of the first 325 positive even whole numbers. Sum=[ANS]
Solution:
There is 1 [ANS] so I need exactly 1 answer.
Sum = 325 * (1 + 325)
\\boxed{325*(1+325)}

Example 2 (INCORRECT — do NOT do this):
Problem: A data set has values -10, -9, 1, 3, 7, 8. Find each value's deviation [ANS] (x6), squared deviation [ANS] (x6), sum [ANS], variance [ANS], std dev [ANS].
WRONG answer (do NOT imitate this):
\\boxed{7.797}   ← WRONG! 15 [ANS] placeholders but only boxed 1 value. Give ALL values with full precision.
CORRECT answer:
\\boxed{-10, 100, -9, 81, 1, 1, 3, 9, 7, 49, 8, 64, 304, 60.8, 7.79743547584717}

Example 3 (INCORRECT — rounding when problem says exact):
Problem: Use logarithms to find an EXACT solution. 7*3^t = 245. t=[ANS] (do NOT approximate)
WRONG answer (do NOT imitate this):
\\boxed{3.298}   ← WRONG! Problem says EXACT and do NOT approximate.
CORRECT answer:
t = ln(245/7)/ln(3) = ln(35)/ln(3)
\\boxed{\\frac{\\ln(35)}{\\ln(3)}}

Example 4 (INCORRECT — "at least X decimal places" means MORE not less):
Problem: Solve for q where 30*(0.8)^q = 18. Give answer accurate to at least 1 decimal place. [ANS]
q = ln(0.6)/ln(0.8) = 2.2892
WRONG answer (do NOT imitate this):
\\boxed{2.3}   ← WRONG! "At least 1 decimal place" means give full precision, not round to 1 place.
CORRECT answer:
\\boxed{2.2892}

Example 5 (CORRECT — confidence interval with parentheses):
Problem: Find the 95% confidence interval. Confidence Interval=[ANS]
Lower=81.03, Upper=84.41. Single [ANS] → wrap as one item with parentheses.
\\boxed{(81.03, 84.41)}

Example 6 (CORRECT — coordinate pair as single answer):
Problem: The other zeros of P(x) are x=[ANS]. (separate by commas if multiple)
P(x) has zeros at x=2 and x=-2. One [ANS] → both zeros wrapped in parentheses as one item.
\\boxed{(2, -2)}

Example 7 (INCORRECT — splitting single [ANS] letter combination):
Problem: Which of the following statements are true? Select ALL that apply. [ANS]
Options: A. ..., B. ..., C. ..., D. ..., E. ..., F. ...
Correct options are C, E, F.
WRONG answer (do NOT imitate this):
\\boxed{C, E, F}   ← WRONG! There is only 1 [ANS]. Multiple letters must be concatenated as ONE item.
CORRECT answer:
\\boxed{CEF}

Example 8 (INCORRECT — splitting single [ANS] numerical set):
Problem: Find the vector v. v=[ANS]
v has components (80, -160, 320).
WRONG answer (do NOT imitate this):
\\boxed{80, -160, 320}   ← WRONG! There is only 1 [ANS]. Must be ONE item wrapped in parentheses.
CORRECT answer:
\\boxed{(80, -160, 320)}

Example 9 (INCORRECT — trailing zeros, currency symbols, units):
Problem: Find the annual salary. [ANS]
Annual salary = $38800.
WRONG answer (do NOT imitate this):
\\boxed{$38,800.00}   ← WRONG! No currency symbols, no trailing zeros, no commas in numbers.
CORRECT answer:
\\boxed{38800}

Example 10 (INCORRECT — infty instead of infinity):
Problem: Find the interval where f is increasing. [ANS]
f increases for x > -8.
WRONG answer (do NOT imitate this):
\\boxed{(-8, \\infty)}   ← WRONG! Write infinity not \\infty or infty or inf.
CORRECT answer:
\\boxed{(-8, infinity)}

Example 11 (CORRECT — YES/NO capitalized, percentage with symbol):
Problem: Is the result significant (YES or NO)? [ANS] What percentage? [ANS]
Result is significant. Percentage = 12.5%.
\\boxed{YES, 12.5%}

Example 12 (CORRECT — scientific notation when asked):
Problem: Express in scientific notation. Answer=[ANS] grams
Mass = 0.0000000000000000000001338 grams = 1.338E-22 grams
\\boxed{1.338E-22}

Now solve the following problem:
"""

FEW_SHOT_MCQ = """Here are examples of how to solve multiple choice problems correctly and incorrectly:

Example 1 (CORRECT):
Problem: The melting speed of half of the spherical snowdrift is directly proportional to the surface area of the hemisphere with proportional coefficient k > 0. The volume of the snowdrift with radius r_0 is the original 1/8 after melting for 3 hours. The time required for full melting is?
Options:
A. 6 hours
B. 9 hours
C. 24 hours
D. 15 hours
E. 3 hours
Solution:
Volume = (2/3)πr³. If volume is 1/8 original, r = r₀/2 after 3 hours.
dr/dt = -k, so r = r₀ - kt. After 3 hours: k = r₀/6.
Full melting at r=0: t = r₀/k = 6 hours.
The answer is option A.
\\boxed{A}

Example 2 (INCORRECT — do NOT do this):
Problem: Let N = number of terms in a sequence. If the sum S = N*(N+1)/2 and N=10, which option gives the correct sum?
Options:
A. 55
B. 60
C. 65
D. 70
E. 75
WRONG solution (do NOT imitate this):
Let N = 10. S = N*(N+1)/2 = 10*11/2 = 55.
\\boxed{N}   ← WRONG! N is a variable name from the working, not an option letter.
CORRECT solution:
Let N = 10. S = N*(N+1)/2 = 10*11/2 = 55. Matches option A.
The answer is option A.
\\boxed{A}

Example 3 (INCORRECT — equivalent options, pick simplified):
Problem: Find the derivative. Which option is correct?
Options:
D. 1/(sin^2(x) * cos^5(x) * cos(x))
E. 1/(sin^2(x) * cos^6(x))
WRONG solution (do NOT imitate this):
The derivative = 1/(sin²x · cos^5(x) · cos(x)).
\\boxed{D}   ← WRONG! D and E are equivalent. Always choose the SIMPLIFIED form.
CORRECT solution:
cos^5(x)*cos(x) = cos^6(x), so E is simpler.
The answer is option E.
\\boxed{E}

Example 4 (INCORRECT — do NOT do this):
Problem: Find the value of ∫₀¹ x·eˣ dx.
Options:
A. e - 1
B. e - 2
C. 1
D. e + 1
E. 2e - 1
WRONG solution (do NOT imitate this):
∫₀¹ x·eˣ dx = e - (e-1) = 1.
\\boxed{E}   ← WRONG! Computed 1 which matches C, not E.
CORRECT solution:
∫₀¹ x·eˣ dx = [x·eˣ]₀¹ - ∫₀¹ eˣ dx = e - (e-1) = 1. Matches C.
The answer is option C.
\\boxed{C}

Now solve the following problem:
"""

def build_prompt(question, options):
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, FEW_SHOT_MCQ + f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, FEW_SHOT_MATH + question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.95,
    max_model_len=131072,
    trust_remote_code=True,
    max_num_seqs=32,
    max_num_batched_tokens=131072,
 )

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    repetition_penalty=1.0,
)

print("Model loaded.")

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# # Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!